In [1]:
print("Hello")

Hello


In [2]:
import nltk
print("Working ✅")

Working ✅


In [3]:
# import libraries
import pandas as pd
import numpy as np
import re
import nltk

from nltk.corpus import stopwords
from nltk.stem import PorterStemmer, WordNetLemmatizer

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer

from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import MultinomialNB
from sklearn.tree import DecisionTreeClassifier

from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

In [4]:
!pip install scikit-learn

Defaulting to user installation because normal site-packages is not writeable


In [5]:
from sklearn.model_selection import train_test_split
print("sklearn working ✅")

sklearn working ✅


In [6]:
# Load dataset 
import pandas as pd

df = pd.read_csv(r"C:\Users\S Mohit\innomatics\archive\IMDB Dataset.csv")
df.head()

,review,sentiment
0,One of the other reviewers has mentioned that ...,positive
1,A wonderful little production. <br /><br />The...,positive
2,I thought this was a wonderful way to spend ti...,positive
3,Basically there's a family where a little boy ...,negative
4,"Petter Mattei's ""Love in the Time of Money"" is...",positive


In [7]:
# data understanding 
df.shape
df['sentiment'].value_counts()
df.sample(5)

,review,sentiment
48011,"Okay, When I bought this flick I though this g...",negative
32202,"This was not a very good movie, the acting pre...",negative
29861,"When I saw that Icon was on TV, I was surprise...",negative
29602,I've seen a lot of TV movies in my time as a s...,positive
27028,Greg Davis and Bryan Daly take some crazed sta...,negative


In [8]:
pip install nltk

Defaulting to user installation because normal site-packages is not writeable
Note: you may need to restart the kernel to use updated packages.


In [9]:
import nltk
nltk.download('stopwords')
nltk.download('wordnet')

[nltk_data] Downloading package stopwords to C:\Users\S
[nltk_data]     Mohit\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package wordnet to C:\Users\S
[nltk_data]     Mohit\AppData\Roaming\nltk_data...
[nltk_data]   Package wordnet is already up-to-date!


True

In [10]:
# create Preprocessing function
stop_words = set(stopwords.words('english'))
lemmatizer = WordNetLemmatizer()

def clean_text(text):
    text = text.lower()  # Lowercasing
    
    text = re.sub(r'http\S+', '', text)  # Remove URLs
    text = re.sub(r'[^a-zA-Z]', ' ', text)  # Remove punctuation
    
    words = text.split()  # Tokenization
    
    words = [word for word in words if word not in stop_words]  # Stopwords
    
    words = [lemmatizer.lemmatize(word) for word in words]  # Lemmatization
    
    return " ".join(words)

In [11]:
# apply preprocessing
df['cleaned_review'] = df['review'].apply(clean_text)

In [12]:
# Feature engineering 
# 1. BoW
bow = CountVectorizer(max_features=5000)
X_bow = bow.fit_transform(df['cleaned_review'])

# 2. TF-IDF
tfidf = TfidfVectorizer(max_features=5000)
X_tfidf = tfidf.fit_transform(df['cleaned_review'])

# 3. labels
y = df['sentiment'].map({'positive':1, 'negative':0})


In [13]:


# Train-Test split
X_train, X_test, y_train, y_test = train_test_split(X_tfidf, y, test_size=0.2)

# Model Building
# 1. logic regression
lr = LogisticRegression()
lr.fit(X_train, y_train)
y_pred_lr = lr.predict(X_test)

# 2. Naive bayes
nb = MultinomialNB()
nb.fit(X_train, y_train)
y_pred_nb = nb.predict(X_test)

# 3. Decision tree
dt = DecisionTreeClassifier()
dt.fit(X_train, y_train)
y_pred_dt = dt.predict(X_test)

# evaluation
def evaluate(y_test, y_pred):
    print("Accuracy:", accuracy_score(y_test, y_pred))
    print("Precision:", precision_score(y_test, y_pred))
    print("Recall:", recall_score(y_test, y_pred))
    print("F1 Score:", f1_score(y_test, y_pred))

# Evaluate models
print("Logistic Regression")
evaluate(y_test, y_pred_lr)

print("Naive Bayes")
evaluate(y_test, y_pred_nb)

print("Decision Tree")
evaluate(y_test, y_pred_dt)



Logistic Regression
Accuracy: 0.8908
Precision: 0.880746561886051
Recall: 0.9023752012882448
F1 Score: 0.8914297076953669
Naive Bayes
Accuracy: 0.8588
Precision: 0.850828729281768
Recall: 0.8679549114331723
F1 Score: 0.859306496612196
Decision Tree
Accuracy: 0.7183
Precision: 0.714456630109671
Recall: 0.7212157809983897
F1 Score: 0.7178202945006511


In [14]:
results = pd.DataFrame({
    'Model': ['LR (TF-IDF)', 'NB (TF-IDF)', 'DT (TF-IDF)'],
    'Accuracy': [
        accuracy_score(y_test, y_pred_lr),
        accuracy_score(y_test, y_pred_nb),
        accuracy_score(y_test, y_pred_dt)
    ],
    'Precision': [
        precision_score(y_test, y_pred_lr),
        precision_score(y_test, y_pred_nb),
        precision_score(y_test, y_pred_dt)
    ],
    'Recall': [
        recall_score(y_test, y_pred_lr),
        recall_score(y_test, y_pred_nb),
        recall_score(y_test, y_pred_dt)
    ],
    'F1 Score': [
        f1_score(y_test, y_pred_lr),
        f1_score(y_test, y_pred_nb),
        f1_score(y_test, y_pred_dt)
    ]
})

results

,Model,Accuracy,Precision,Recall,F1 Score
0,LR (TF-IDF),0.8908,0.880747,0.902375,0.891430
1,NB (TF-IDF),0.8588,0.850829,0.867955,0.859306
2,DT (TF-IDF),0.7183,0.714457,0.721216,0.717820


### Insights

- Logistic Regression performed best among all models.
- TF-IDF gave better performance than Bag of Words because it focuses on important words.
- Naive Bayes was fast but slightly less accurate.
- Decision Tree showed lower performance due to overfitting.
- Preprocessing (stopwords removal + lemmatization) improved model accuracy.

### Conclusion

In this project, we built a complete NLP pipeline for sentiment analysis.

- Raw text was cleaned using preprocessing techniques.
- Text was converted into numerical form using TF-IDF and BoW.
- Multiple models were trained and evaluated.

Final Results:
- Best Model: Logistic Regression
- Best Feature Technique: TF-IDF

This project shows how raw text can be transformed into meaningful predictions using NLP and Machine Learning.